# Notebook 02 — TextCNN From Scratch
**Model:** TextCNN | **Metric:** MAP@3

In [20]:
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import wandb
from kaggle_secrets import UserSecretsClient

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [21]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test= pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
OPTIONS = ['A', 'B', 'C', 'D', 'E']
print('Train:', train.shape, '| Test:', test.shape)

Train: (2000, 8) | Test: (500, 7)


In [22]:
def clean(t):
    return re.sub(r'[^a-z0-9 ]', '', str(t).lower())

all_text = ' '.join(
    train['prompt'].apply(clean).tolist() +
    train['A'].apply(clean).tolist() +
    train['B'].apply(clean).tolist() +
    train['C'].apply(clean).tolist() +
    train['D'].apply(clean).tolist() +
    train['E'].apply(clean).tolist()
)
vocab = ['<PAD>', '<UNK>'] + list(set(all_text.split()))
w2i = {w: i for i, w in enumerate(vocab)}

def encode(text, maxlen=64):
    words = clean(text).split()
    ids = [w2i.get(w, 1) for w in words]
    ids = ids[:maxlen]

    return ids + [0] * (maxlen - len(ids))

print('Vocab size:', len(vocab))

Vocab size: 3084


In [23]:
#encode prompt + all 5 options concatenated
def encode_row(row, maxlen=64):
    text = row['prompt'] + ' ' + ' '.join([str(row[o]) for o in OPTIONS])
    return encode(text, maxlen)

# Encode every training row
encoded_rows = []

for _, row in train.iterrows():
    encoded_row = encode_row(row)
    encoded_rows.append(encoded_row)

X = torch.tensor(encoded_rows, dtype=torch.long)


# Convert answers A–E into numbers 0–4
answer_labels = []

for answer in train["answer"]:
    label = ord(answer) - ord("A")
    answer_labels.append(label)

y = torch.tensor(answer_labels, dtype=torch.long)

print('X:', X.shape, '| y:', y.shape)

X: torch.Size([2000, 64]) | y: torch.Size([2000])


In [24]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

X_train = X_train.to(device)
y_train = y_train.to(device)

X_val = X_val.to(device)
y_val = y_val.to(device)

print("Train:", X_train.shape, "| Val:", X_val.shape)

Train: torch.Size([1700, 64]) | Val: torch.Size([300, 64])


In [25]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.conv = nn.Conv1d(64, 64, kernel_size=3)
        self.fc = nn.Linear(64, 5) 

    def forward(self, x):
        x = self.emb(x).permute(0, 2, 1) # (batch, embed, seq)
        x = F.relu(self.conv(x)) # convolution
        x = F.max_pool1d(x, x.shape[2]).squeeze(2) # max pooling
        return self.fc(x) # output 5 scores

model = TextCNN(len(vocab)).to(device)
print(model)

TextCNN(
  (emb): Embedding(3084, 64, padding_idx=0)
  (conv): Conv1d(64, 64, kernel_size=(3,), stride=(1,))
  (fc): Linear(in_features=64, out_features=5, bias=True)
)


In [26]:
def compute_map3(logits, labels):
    probs= torch.softmax(logits, dim=1).cpu().numpy()
    correct = labels.cpu().numpy()
    aps = []
    for i in range(len(correct)):
        top3 = probs[i].argsort()[::-1][:3].tolist()
        ap, hits = 0.0, 0
        for rank, idx in enumerate(top3):
            if idx == correct[i]:
                hits += 1
                ap+= hits / (rank + 1)
        aps.append(ap)
    return float(np.mean(aps))

In [27]:
secret = UserSecretsClient().get_secret('GENAI_T2_2026')
wandb.login(key=secret)
wandb.init(
    project = '23f2003665-t22026',
    name = 'textcnn-final',
    config= {
        'vocab_size': len(vocab),
        'embed_dim': 64,
        'num_filters': 64,
        'kernel_size': 3,
        'lr' : 1e-3,
        'epochs': 10,
    }
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
best_map3 = 0.0

for epoch in range(10):
    # Train
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(X_train), y_train)
    loss.backward()
    optimizer.step()

    # Validate
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
    val_map3 = compute_map3(val_logits, y_val)

    print(f'Epoch {epoch+1:>2} | Loss: {loss.item():.4f} | Val MAP@3: {val_map3:.4f}')
    wandb.log({'epoch': epoch+1, 'train_loss': loss.item(), 'val_map3': val_map3})

    # Save best model
    if val_map3 > best_map3:
        best_map3 = val_map3
        torch.save(model.state_dict(), '/kaggle/working/textcnn_best.pt')
        print(f'Best saved (MAP@3={best_map3:.4f})')

wandb.log({'best_val_map3': best_map3})
wandb.finish()
print('Training complete. Best MAP@3:', best_map3)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch  1 | Loss: 1.8975 | Val MAP@3: 0.3911
Best saved (MAP@3=0.3911)
Epoch  2 | Loss: 1.7844 | Val MAP@3: 0.4061
Best saved (MAP@3=0.4061)
Epoch  3 | Loss: 1.6868 | Val MAP@3: 0.4150
Best saved (MAP@3=0.4150)
Epoch  4 | Loss: 1.6051 | Val MAP@3: 0.4517
Best saved (MAP@3=0.4517)
Epoch  5 | Loss: 1.5401 | Val MAP@3: 0.5367
Best saved (MAP@3=0.5367)
Epoch  6 | Loss: 1.4917 | Val MAP@3: 0.6067
Best saved (MAP@3=0.6067)
Epoch  7 | Loss: 1.4580 | Val MAP@3: 0.5972
Epoch  8 | Loss: 1.4354 | Val MAP@3: 0.5678
Epoch  9 | Loss: 1.4189 | Val MAP@3: 0.5594
Epoch 10 | Loss: 1.4040 | Val MAP@3: 0.5550


best_val_map3,▁
epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▆▅▄▃▂▂▁▁▁
val_map3,▁▁▂▃▆██▇▆▆
best_val_map3,0.60667
epoch,10
train_loss,1.40397
val_map3,0.555


Training complete. Best MAP@3: 0.6066666666666667


In [28]:
# Load best checkpoint
model.load_state_dict(torch.load('/kaggle/working/textcnn_best.pt'))
model.eval()

#encode test data
X_test = torch.tensor(
    [encode_row(row) for _, row in test.iterrows()],
    dtype=torch.long
).to(device)

with torch.no_grad():
    outputs = model(X_test)
    probs = torch.softmax(outputs, dim=1).cpu().numpy()

predictions = []

for probability in probs:
    sorted_indices = probability.argsort()[::-1]
    top3_indices = sorted_indices[:3]

    top3_options = [OPTIONS[index] for index in top3_indices]
    top3_prediction = " ".join(top3_options)

    predictions.append(top3_prediction)

submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

submission["Prediction"] = predictions
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Submission saved!")
submission.head()

Submission saved!


,ID,Prediction
0,1,A B C
1,2,C B E
2,3,B C E
3,4,E B A
4,5,C B A


**Kaggle score :** 0.45178